# Instalación e importación de librerías

Si estas en colab:

In [ ]:
!pip install scikit-image opencv-python matplotlib numpy imageio Pillow tqdm
!pip install fvcore iopath
!pip install 'git+https://github.com/facebookresearch/pytorch3d.git@main'

In [1]:
import os
import sys
import torch

In [2]:
import glob
import numpy as np
from tqdm import tqdm
import imageio
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from skimage import img_as_ubyte
from pytorch3d.utils import ico_sphere
import argparse

# io utils
from pytorch3d.io import load_obj, save_obj
from pytorch3d.ops import cubify

# datastructures
from pytorch3d.structures import Meshes, Volumes

# 3D transformations functions
from pytorch3d.transforms import Rotate, Translate

import random

# rendering components
from pytorch3d.renderer import (
    FoVPerspectiveCameras,
    FoVOrthographicCameras,
    VolumeRenderer,
    NDCGridRaysampler,
    EmissionAbsorptionRaymarcher,
    look_at_view_transform,
    TexturesVertex
)

from pytorch3d.loss import (
    chamfer_distance,
    mesh_edge_loss,
    mesh_laplacian_smoothing,
    mesh_normal_consistency,
)

import cv2
from utills import (get_voxel_renderer,get_phong_renderer, create_cameras, create_cameras_TFS_mode, create_cameras_4VTFS_mode, render_voxels)

from datasets import load_data, load_data_from_list
from losses import (huber, silh_loss, MS_SSIM, l1_loss, iou_np, dice_np)
from models import VolumeModel


# Función de entrenamiento

In [3]:
def train():

    global exp_sample_id
    #############################################################
    #             Setting the rendering parameters              #
    #############################################################

    if TFS_mode:
        cameras, Rs, Ts = create_cameras_TFS_mode(device, zdist, mirror_mode, camera_mode)
    elif FourVTSF_mode:
        cameras, Rs, Ts = create_cameras_4VTFS_mode(device, zdist, mirror_mode, camera_mode)
    else:
        cameras, Rs, Ts = create_cameras(num_views, device, zdist, camera_mode, mirror_mode)

    renderer = get_voxel_renderer(device, cameras, img_size, volume_extent_world)
    phong_renderer = get_phong_renderer(device, FoVOrthographicCameras(device=device), img_size)

    i = 0

    print("Experiment:",main_exp_id+sub_exp_id)
    print("Sample : ",i)

    try:
        os.mkdir(root_dir+main_exp_id)
    except:
        print("Directory already exists")

    try:
        os.mkdir(root_dir+main_exp_id+sub_exp_id)
    except:
        print("Directory already exists")

    output_vid_path = root_dir+main_exp_id+sub_exp_id+"/sample_%d_vid.gif"%i
    print(output_vid_path)
    writer = imageio.get_writer(output_vid_path, mode='I', duration=0.1)

    silhs, silhs_tensors = load_data_from_list(shadow_files,
            mirror_mode,
            (img_size, img_size),
            device,
            num_views,
            debug_mode
            )

    volume_size = 128 # Voxel Resolution
    volume_model = VolumeModel(
        renderer,
        volume_size=[volume_size] * 3,
        voxel_size = volume_extent_world / volume_size,
        thresh_density = thresh_density
    ).to(device)

    optimizer = torch.optim.Adam(volume_model.parameters(), lr=lr)
    batch_size = 1

    loop = tqdm(range(Niter))
    # loop = range(Niter)

    ms_ssim_list = []

    # for i in loop:
    for iteration in loop:
        print(iteration)

        # In case we reached the last 75% of iterations,
        # decrease the learning rate of the optimizer 10-fold.
        if iteration == round(Niter * 0.75):
            print('Decreasing LR 10-fold ...')
            optimizer = torch.optim.Adam(
                volume_model.parameters(), lr=lr * 0.1
            )

        # Sample random batch indices.
        # batch_idx = torch.randperm(len(cameras))[:batch_size]
        # print(len(cameras))
        for batch_idx in range(len(cameras)):

            # Zero the optimizer gradient.
            optimizer.zero_grad()

            if camera_mode == "ortho":
                batch_cameras = FoVOrthographicCameras(
                    R = cameras.R[batch_idx].unsqueeze(0),
                    T = cameras.T[batch_idx].unsqueeze(0),
                    znear = cameras.znear[batch_idx].unsqueeze(0),
                    zfar = cameras.zfar[batch_idx].unsqueeze(0),
                    device = device,
                )
            else:
                batch_cameras = FoVPerspectiveCameras(
                    R = cameras.R[batch_idx].unsqueeze(0),
                    T = cameras.T[batch_idx].unsqueeze(0),
                    znear = cameras.znear[batch_idx].unsqueeze(0),
                    zfar = cameras.zfar[batch_idx].unsqueeze(0),
                    device = device,
                )

            # Evaluate the volumetric model.
            rendered_images, rendered_silhouettes = volume_model(
                batch_cameras
            ).split([3, 1], dim=-1)

            pred_output = rendered_images[0][:,:,0]

            sil_err = silh_loss(
                pred_output, silhs_tensors[batch_idx],
            )

            l1_err = l1_loss(
                pred_output.view(1,img_size,img_size).type(torch.float32).to(device),
                silhs_tensors[batch_idx].view(1,img_size,img_size).type(torch.float32).to(device)
            )

            ms_ssim_err = ms_ssim_loss(
                pred_output.view(1,1,img_size,img_size).type(torch.float32).to(device),
                silhs_tensors[batch_idx].view(1,1,img_size,img_size).type(torch.float32).to(device)
            )

            loss = sil_err *silh_wt + l1_err*l1_wt + ms_ssim_err*ms_ssim_wt

            if iteration%10 == 0:
                print(
                        f'Iteration {iteration:04d}:'
                        + f' loss = {float(loss):1.2e}'
                    )

            # Take the optimization step.
            loss.backward()
            optimizer.step()
            silh_pth = root_dir + main_exp_id + sub_exp_id + "/sample_%dview_%d_silh.png"%(exp_sample_id,batch_idx)
            silh_view_img = (pred_output.detach().cpu().numpy()*255).astype(np.uint8)
            ret, silh_view_img = cv2.threshold(silh_view_img,np.max(silh_view_img)*0.8,255,cv2.THRESH_BINARY)
            cv2.imwrite(silh_pth,silh_view_img)

            ms_ssim_list.append(ms_ssim_err.item())

        if iteration%10 == 0:
            print(
                    f'Iteration {iteration:04d}:'
                    + f' loss = {float(loss):1.2e}'
                )
            R, T = look_at_view_transform(zdist, 0, iteration, device=device)
            volumes = Volumes(
                densities = volume_model.voxels[None].expand(
                    batch_size, *volume_model.log_densities.shape),
                features = volume_model.colors[None].expand(
                    batch_size, *volume_model.log_colors.shape),
                voxel_size=volume_model._voxel_size,
            )
            image, silhouette = renderer(cameras=FoVOrthographicCameras(R=R, T=T, device=device), volumes=volumes)[0].split([3, 1], dim=-1)
            image = image[0, ..., :3].detach().squeeze().cpu().numpy()
            image = img_as_ubyte(image)
            writer.append_data(image)

    writer.close()

    mesh1 = cubify(volume_model.voxels,thresh_density)
    final_verts = mesh1.verts_packed()
    final_faces = mesh1.faces_packed()

    # # Store the predicted mesh using save_obj
    final_obj_pth = root_dir + main_exp_id + sub_exp_id +"/sample_%d_output.obj"%exp_sample_id
    final_voxel_pth = root_dir + main_exp_id + sub_exp_id +"/sample_%d_final_voxels.npy"%exp_sample_id

    save_obj(final_obj_pth, final_verts, final_faces)

    voxels = volume_model.voxels.detach().cpu().numpy()
    colors = volume_model.colors.detach().cpu().numpy()

    with open(final_voxel_pth, 'wb') as f:
        np.save(f,voxels)

    folder_pth = root_dir + main_exp_id + sub_exp_id
    for i, img in enumerate(silhs):
        cv2.imwrite(folder_pth+"/sample_%dview_%d.png"%(exp_sample_id,i),img)

    if debug_mode:
        with open(final_voxel_pth, 'rb') as f:
            voxels = np.load(f)
        render_voxels(voxels, volume_extent_world, volume_size, zdist, renderer, device)

    ms_ssim_metric = 1 - np.array(ms_ssim_list).mean()

    mean_iou = 0.0
    mean_dice = 0.0
    for idx in range(num_views*(1+mirror_mode)):

        gt_shadow = cv2.imread(folder_pth+"/sample_%dview_%d.png"%(exp_sample_id,idx),0)
        pred_shadow = cv2.imread(folder_pth+"/sample_%dview_%d_silh.png"%(exp_sample_id,idx),0)
        diff_img = np.abs(gt_shadow - pred_shadow)

        mean_iou+= iou_np(gt_shadow,pred_shadow)
        mean_dice+= dice_np(gt_shadow, pred_shadow)

        gt_shadow = cv2.bitwise_not(gt_shadow)
        pred_shadow = cv2.bitwise_not(pred_shadow)

        cv2.imwrite(folder_pth+"/SHADOW_GT%d_view_%d.png"%(exp_sample_id,idx),gt_shadow)
        cv2.imwrite(folder_pth+"/SHADOW_PRED%d_view_%d.png"%(exp_sample_id,idx),pred_shadow)
        cv2.imwrite(folder_pth+"/SHADOW_DIFF%d_view_%d.png"%(exp_sample_id,idx),diff_img)



    mean_iou = mean_iou/(1.0*num_views*(1+mirror_mode))
    mean_dice = mean_dice/(1.0*num_views*(1+mirror_mode))

    text = ""
    text += "\nEdge loss : " + str(mesh_edge_loss(mesh1).item())
    text += "\nLaplacian loss : " + str(mesh_laplacian_smoothing(mesh1).item())
    text += "\nNormal loss : " + str(mesh_normal_consistency(mesh1).item())
    text += "\nIOU metric: " + str(mean_iou)
    text += "\nDice metric: " + str(mean_dice)
    text += "\nMISSIM metric: " + str(ms_ssim_metric)

    text_file = open(folder_pth+"/log.txt", "w")
    n = text_file.write(cmd_input + text)
    text_file.close()

    exp_sample_id+=1

# Inputs y ejecución

In [ ]:
parser = argparse.ArgumentParser(description = "List of various parameters for experiments")
parser.add_argument("device", type=str, help="GPU number")
parser.add_argument("sub_exp_id", type=str, help="sub experiment id")
parser.add_argument("Niter", type=int, help="Number of iterations")
parser.add_argument("lr", type=float, help="Learning rate")

parser.add_argument("-vfl","--views_file_name", type=str, help="Name of file containing path to ground truth views",default="dataset1.txt")
parser.add_argument("-mr","--mirror_mode", type=bool, help="Mirror mode set to true if front and rear both views are to be regressed", default=0)
parser.add_argument("-mr2","--mirror_mode_2", type=bool, help="Mirror mode set to true if front and rear both views are to be regressed", default=0)
parser.add_argument("-tsf","--TSF_mode", type=bool, help="set true for Top-Side-Front view 3 view setup", default=0)
parser.add_argument("-tsf4","--TSF4V_mode", type=bool, help="set true for TSF with three side view setup", default=0)
parser.add_argument("-cam","--camera_mode", type=str, help="set camera mode as ortho or perspective", default="ortho")
parser.add_argument("-imsz","--img_size", type=int, help="set image size",default=512)
parser.add_argument("-swt","--silh_wt", type=float, help="Silhoutte loss weight", default=10.0)
parser.add_argument("-l1wt","--l1_wt", type=float, help="L1 loss weight", default=10.0)
parser.add_argument("-mwt","--ms_ssim_wt", type=float, help="MS_SSIM loss weight", default=0.0)
parser.add_argument("-ns","--num_samples", type=int, help="Number of samples", default=1)
parser.add_argument("-th","--thresh_density", type=float, help="Cubify function threshold", default=0.05)
parser.add_argument("-zd","--zdist", type=float, help="Cubify function threshold", default=1.7)
parser.add_argument("-sdlist", "--shadow_files", nargs="+", default=["None"])

output = "10-5media-5alta"
args = parser.parse_args([
    "cuda:0",
    output,
    "600",                       # Iteraciones completas
    "0.01",                     # Learning rate preciso para voxeles
    "-imsz", "512",
    "-swt", "10.0",
    "-l1wt", "10.0",
    "-sdlist", "fish.png", "bird.png", "house.png", "tree.png", "sitting_cat.png", "running_person.png", "guitar.png", "butterfly.png", "bike.png", "snowflake.png"
])

In [ ]:
#############################################################
#                 Experiment Key Parameters                 #
#############################################################

# setting Device
if torch.cuda.is_available() and (args.device != "cpu"):
    device = torch.device(args.device)
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")

print("Device: ", device)

ms_ssim_loss = MS_SSIM(device)

random.seed(43)
root_dir = "/"
main_exp_id = "voxel_results/"

sub_exp_id = args.sub_exp_id
file_name = args.views_file_name
mirror_mode = False
FourVTSF_mode = False
mirror_mode_2 = False # Make it false to get cameras as mirror view but rear view as not a mirror view but other obj view
thresh_density = args.thresh_density
TFS_mode = False
camera_mode = args.camera_mode
img_size = args.img_size
Niter = args.Niter
zdist = args.zdist
debug_mode = False
num_samples = args.num_samples
volume_extent_world = 1.7
exp_sample_id = 0
lr = args.lr
l1_wt = args.l1_wt
silh_wt = args.silh_wt
ms_ssim_wt = args.ms_ssim_wt
shadow_files = args.shadow_files
num_views = len(shadow_files)


cmd_input = "The command line input string \n"+str(sys.argv)

# python train.py cuda:1 temp_trial 30 0.01 -swt 10.0 -l1wt 10.0 -mwt 0.0 -ns 2
# python val.py cuda:0 output1 600 0.01 -swt 10.0 -l1wt 10.0 -sdlist duck.png mikey.png

Device:  cuda:0


In [ ]:
# 1. Definir la lista de todos los experimentos según la nueva nomenclatura
experimentos_and_baseline = [
    # Experimentos espejo de generalización
    {
        "output": "2a-media-alta",
        "num_views": 2,
        "images": ["m1_bird.png", "h2_running_person.png"]
    },
    {
        "output": "3b-1media-2alta",
        "num_views": 3,
        "images": ["m5_guitar.png", "h1_sitting_cat.png", "h3_butterfly.png"]
    },
    {
        "output": "3c-2simple-1media",
        "num_views": 3,
        "images": ["s5_house.png", "s4_flecha.png", "m3_fish.png"]
    },
    {
        "output": "3d-simple-media-alta",
        "num_views": 3,
        "images": ["s2_rectangulo.png", "m1_bird.png", "h2_running_person.png"]
    },

    # Experimentos de capacidad
    {
        "output": "2-simple-simple",
        "num_views": 2,
        "images": ["s1_circulo.png", "s3_cruz.png"]
    },
    {
        "output": "2-simple-alta",
        "num_views": 2,
        "images": ["s2_rectangulo.png", "h2_running_person.png"]
    },
    {
        "output": "2-alta-alta",
        "num_views": 2,
        "images": ["h3_snowflake.png", "h5_bike.png"]
    },
    {
        "output": "5-simple",
        "num_views": 5,
        "images": ["s1_circulo.png", "s2_rectangulo.png", "s3_cruz.png", "s4_flecha.png", "s5_house.png"]
    },
    {
        "output": "5-media",
        "num_views": 5,
        "images": ["m1_bird.png", "m2_tree.png", "m3_fish.png", "m4_star.png", "m5_guitar.png"]
    },
    {
        "output": "5-alta",
        "num_views": 5,
        "images": ["h1_sitting_cat.png", "h2_running_person.png", "h3_snowflake.png", "h4_butterfly.png", "h5_bike.png"]
    },
    {
        "output": "5-2simple-2media-1alta",
        "num_views": 5,
        "images": ["s1_circulo.png", "s5_house.png", "m1_bird.png", "m5_guitar.png", "h3_snowflake.png"]
    },
    {
        "output": "10-5simple-5media",
        "num_views": 10,
        "images": [
            "s1_circulo.png", "s2_rectangulo.png", "s3_cruz.png", "s4_flecha.png", "s5_house.png",
            "m1_bird.png", "m2_tree.png", "m3_fish.png", "m4_star.png", "m5_guitar.png"
        ]
    },
    {
        "output": "10-5media-5alta",
        "num_views": 10,
        "images": [
            "m1_bird.png", "m2_tree.png", "m3_fish.png", "m4_star.png", "m5_guitar.png",
            "h1_sitting_cat.png", "h2_running_person.png", "h3_snowflake.png", "h4_butterfly.png", "h5_bike.png"
        ]
    },

    # Baseline, figura 4 paper
    {
        "output": "2-mikey-puma",
        "num_views": 2,
        "images": ["mikey.png", "puma.png"]
    },
    {
        "output": "3-heroes",
        "num_views": 3,
        "images": ["Spider-Man.png", "superman.png", "batman.png"]
    },
    {
        "output": "3-bunny-teddy-duck",
        "num_views": 3,
        "images": ["bunny_0.png", "teddy2.png", "duck.png"]
    },
    {
        "output": "3-mikey-puma-heart",
        "num_views": 3,
        "images": ["mikey.png", "puma.png", "heart.png"]
    }
]

# 2. Bucle de automatización
for exp in experimentos_and_baseline:
    output_name = exp["output"]
    img_list = exp["images"]

    print("\n" + "="*60)
    print(f" INICIANDO EXPERIMENTO: {output_name}")
    print(f" SILUETAS: {img_list}")
    print("="*60 + "\n")

    # Construir la lista de argumentos simulando la terminal
    cmd_args = [
        "cuda:0",
        output_name,
        "600",         # Niter
        "0.01",        # lr
        "-imsz", "512",
        "-swt", "10.0",
        "-l1wt", "10.0",
        "-sdlist"
    ] + img_list

    # Parsear los nuevos argumentos
    args = parser.parse_args(cmd_args)

    # 3. Actualizar explícitamente las variables globales que usa train()
    # Si estás en un entorno como Colab o Jupyter, actualizar estas variables
    # en el bloque principal afectará el comportamiento de train()
    sub_exp_id = args.sub_exp_id
    shadow_files = args.shadow_files
    num_views = len(shadow_files)

    # Actualizamos hiperparámetros por si acaso, manteniendo consistencia
    Niter = args.Niter
    lr = args.lr
    img_size = args.img_size
    silh_wt = args.silh_wt
    l1_wt = args.l1_wt

    # IMPORTANTE: Reiniciar el ID de la muestra para que no se sobrescriban o nombren mal los outputs
    exp_sample_id = 0

    # Ejecutar el proceso de optimización para esta configuración
    train()

print("\n" + "="*60)
print(" TODOS LOS EXPERIMENTOS HAN FINALIZADO EXITOSAMENTE")
print("="*60)


 INICIANDO EXPERIMENTO: 2-simple-simple
 SILUETAS: ['s1_circulo.png', 's3_cruz.png']

Experiment: voxel_results/2-simple-simple
Sample :  0
/voxel_results/2-simple-simple/sample_0_vid.gif


  0%|          | 0/600 [00:00<?, ?it/s]

0


/tmp/ipykernel_1195/3544796242.py:124: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  + f' loss = {float(loss):1.2e}'


Iteration 0000: loss = 5.37e+00
Iteration 0000: loss = 5.69e+00
Iteration 0000: loss = 5.69e+00


...

 28%|██▊       | 165/600 [03:45<09:58,  1.38s/it]

165


 28%|██▊       | 166/600 [03:46<09:54,  1.37s/it]

166


 28%|██▊       | 167/600 [03:48<09:50,  1.36s/it]

# Modelos 3D y metricas

In [ ]:
def compute_all_metrics(folder, num_views, mirror_mode=0):
    results = []

    for idx in range(num_views * (1 + mirror_mode)):
        gt_path = f"{folder}/sample_0view_{idx}.png"
        pred_path = f"{folder}/sample_0view_{idx}_silh.png"

        gt = cv2.imread(gt_path, 0)
        pred = cv2.imread(pred_path, 0)

        # 1. Binarización estricta de AMBAS imágenes (GT y Predicción)
        # Asegura que cualquier ruido de borde se consolide en 0 o 255 absolutos
        _, gt_bin = cv2.threshold(gt, 127, 255, cv2.THRESH_BINARY)
        _, pred_bin = cv2.threshold(pred, 127, 255, cv2.THRESH_BINARY)

        # 2. Normalización a floats [0.0, 1.0]
        gt_norm   = gt_bin / 255.0
        pred_norm = pred_bin / 255.0

        # 3. Intercepción de NaN (manejo de división por cero aislado de losses.py)
        # Si ambas proyecciones son completamente negras, la coincidencia es del 100%
        if np.sum(gt_norm) == 0 and np.sum(pred_norm) == 0:
            iou_val  = 1.0
            dice_val = 1.0
        else:
            iou_val  = float(iou_np(pred_norm, gt_norm))
            dice_val = float(dice_np(pred_norm, gt_norm))

        # Métricas adicionales desde la matriz de confusión usando booleanos puros
        gt_bool   = gt_bin > 0
        pred_bool = pred_bin > 0

        tp = np.logical_and( gt_bool,  pred_bool).sum()
        fp = np.logical_and(~gt_bool,  pred_bool).sum()
        fn = np.logical_and( gt_bool, ~pred_bool).sum()
        tn = np.logical_and(~gt_bool, ~pred_bool).sum()

        precision = float(tp / (tp + fp)) if (tp + fp) > 0 else 1.0
        recall    = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0

        # ROI Pixel Accuracy sobre bounding box unión
        roi_acc = float(_roi_pixel_accuracy(gt_bool, pred_bool, margin=5))

        results.append({
            "view":      idx,
            "IoU":       iou_val,
            "Dice":      dice_val,
            "Precision": precision,
            "Recall":    recall,
            "ROI_PA":    roi_acc,
        })

    return results

def _roi_pixel_accuracy(gt_bool, pred_bool, margin=5):
    def bbox(mask):
        ys, xs = np.where(mask)
        if len(xs) == 0:
            return None
        return xs.min(), xs.max(), ys.min(), ys.max()

    b_gt   = bbox(gt_bool)
    b_pred = bbox(pred_bool)
    if b_gt is None and b_pred is None:
        return 1.0
    b_gt   = b_gt   or b_pred
    b_pred = b_pred or b_gt

    xmin = max(0, min(b_gt[0], b_pred[0]) - margin)
    xmax = min(gt_bool.shape[1]-1, max(b_gt[1], b_pred[1]) + margin)
    ymin = max(0, min(b_gt[2], b_pred[2]) - margin)
    ymax = min(gt_bool.shape[0]-1, max(b_gt[3], b_pred[3]) + margin)

    gt_roi   = gt_bool  [ymin:ymax+1, xmin:xmax+1]
    pred_roi = pred_bool[ymin:ymax+1, xmin:xmax+1]
    return float(np.mean(gt_roi == pred_roi))

In [ ]:
todos_los_resultados = []

for exp in experimentos_and_baseline:
    output = exp["output"]
    num_views = exp["num_views"]
    images_list = exp["images"]
    folder = f"/voxel_results/{output}"

    print(f"\n{'='*80}")
    print(f" EVALUANDO EXPERIMENTO: {output.upper()}")
    print(f"{'='*80}")

    voxel_path = f"{folder}/sample_0_final_voxels.npy"

    if not os.path.exists(voxel_path):
        print(f"⚠️ Archivo no encontrado: {voxel_path}. Saltando experimento...")
        continue

    # ---------------------------------------------------------
    # A) RENDERIZADO 3D
    # ---------------------------------------------------------
    v = np.load(voxel_path)[0]  # (128,128,128)
    thresh = 0.3
    xs, ys, zs = np.where(v > thresh)

    # Manejo de caso límite: Si el modelo desapareció durante la optimización
    if len(xs) == 0:
        print("⚠️ El modelo 3D colapsó completamente (0 vóxeles superan el umbral). No se puede graficar.")
    else:
        c_norm = (v[xs, ys, zs] - v[xs, ys, zs].min()) / (v[xs, ys, zs].max() - v[xs, ys, zs].min() + 1e-8)

        fig = plt.figure(figsize=(16, 5))
        fig.patch.set_facecolor('#0e1117')

        for i, (title, elev, azim) in enumerate([
            ('Vista frontal',   0,  0),
            ('Vista lateral',   0, 90),
            ('Vista superior', 90,  0),
            ('Vista 3/4',      25, 45),
        ]):
            ax = fig.add_subplot(1, 4, i+1, projection='3d')
            ax.set_facecolor('#0e1117')
            ax.scatter(xs, ys, zs, c=c_norm, cmap='plasma', s=0.8, alpha=0.55, linewidths=0)
            ax.view_init(elev=elev, azim=azim)
            ax.set_xlim(0, 128)
            ax.set_ylim(0, 128)
            ax.set_zlim(0, 128)

            # Forzar cubo y limpiar visualmente
            ax.set_box_aspect((1, 1, 1))
            ax.axis('off')
            ax.set_title(title, color='white', fontsize=10, pad=10)

        fig.suptitle(f'Modelo 3D Voxel — {output}', color='white', fontsize=14, y=0.95)
        plt.tight_layout()
        plt.show()

    # ---------------------------------------------------------
    # B) CÁLCULO Y REGISTRO DE MÉTRICAS
    # ---------------------------------------------------------
    try:
        results = compute_all_metrics(folder, num_views)

        metrics = ["IoU", "Dice", "Precision", "Recall", "ROI_PA"]
        header = f"{'Vista':<20}" + "".join(f"{m:>15}" for m in metrics)
        print(header)
        print("-" * len(header))

        for r in results:
            r['view'] = images_list[r['view']][:-4]
            row = f"{r['view']:<20}" + "".join(f"{r[m]:>15.4f}" for m in metrics)
            print(row)

            # Guardar en memoria para el DataFrame final
            row_data = {"experimento": output, **r}
            todos_los_resultados.append(row_data)

    except Exception as e:
        print(f"⚠️ Error al calcular métricas para {output}: {e}")

# ---------------------------------------------------------
# C) CONSOLIDACIÓN FINAL EN CSV
# ---------------------------------------------------------
if todos_los_resultados:
    df_metricas = pd.DataFrame(todos_los_resultados)

    # Reordenar columnas para que el nombre del experimento sea la primera
    cols = ['experimento', 'view', 'IoU', 'Dice', 'Precision', 'Recall', 'ROI_PA']
    df_metricas = df_metricas[cols]

    csv_path = "/voxel_results/metricas_voxel.csv"
    df_metricas.to_csv(csv_path, index=False)
    print(f"\n✅ Proceso finalizado. Todas las métricas han sido tabuladas y exportadas a: {csv_path}")

Vista            IoU        Dice   Precision      Recall      ROI_PA
--------------------------------------------------------------------
0             0.6392      0.7799      0.9998      0.6392      0.6603
1             0.8335      0.9092      0.8685      0.9538      0.9090
2             0.7165      0.8348      0.9177      0.7657      0.8152
3             0.7207      0.8377      0.8442      0.8313      0.8447
4             0.6039      0.7530      0.7109      0.8005      0.8423
5             0.7673      0.8683      0.8438      0.8943      0.9045
6             0.4823      0.6508      0.6615      0.6404      0.7239
7             0.6222      0.7671      0.7470      0.7884      0.8415
8             0.5235      0.6872      0.5723      0.8599      0.8337
9             0.7318      0.8451      0.8130      0.8799      0.8653
